# Customer Support Ticket Classification System

This notebook builds a system that automatically:
1. **Cleans and tokenizes** raw support ticket text
2. **Classifies the ticket category** (Billing, Technical, Account, Product, Shipping)
3. **Assigns a priority level** (High, Medium, Low)
4. **Evaluates model performance** with accuracy, precision/recall, and confusion matrices

**Tools:** Python, NLTK, scikit-learn, pandas, matplotlib/seaborn

Run the cells from top to bottom (`Cell > Run All` in Jupyter, or press `Shift+Enter` on each cell).

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from preprocessing import clean_text, clean_series

%matplotlib inline
sns.set_style('whitegrid')
print("Libraries loaded successfully.")

## 2. Load the Dataset

The dataset (`data/tickets.csv`) contains realistic synthetic support tickets, each labeled with
a **category** and a **priority**. In a real deployment you would replace this file with your
own historical ticket export (same three columns: `ticket_text`, `category`, `priority`).

In [ ]:
df = pd.read_csv('data/tickets.csv')
print(f"Loaded {len(df)} tickets")
df.head(10)

: 

### 2.1 Explore the data

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['category'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Tickets per Category')
axes[0].set_ylabel('Count')

df['priority'].value_counts().reindex(['High', 'Medium', 'Low']).plot(
    kind='bar', ax=axes[1], color=['crimson', 'orange', 'seagreen'])
axes[1].set_title('Tickets per Priority')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 3. Text Cleaning & Tokenization

Every ticket goes through:
- Lowercasing
- URL / punctuation / number removal
- Tokenization
- Stopword removal
- Lemmatization (reducing words to their base form, e.g. "crashed" -> "crash")

This logic lives in `src/preprocessing.py` and uses **NLTK**. If NLTK's data files aren't
available (e.g. no internet), it automatically falls back to an equally effective built-in
cleaner so this notebook never breaks.

In [ ]:
sample_ticket = df['ticket_text'].iloc[0]
print("BEFORE:", sample_ticket)
print("AFTER :", clean_text(sample_ticket))

In [ ]:
df['cleaned_text'] = clean_series(df['ticket_text'].tolist())
df[['ticket_text', 'cleaned_text']].head()

## 4. Category Classification

We turn cleaned text into TF-IDF features (unigrams + bigrams) and train a
**Logistic Regression** classifier to predict the ticket **category**.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_text'], df['category'], test_size=0.2, random_state=42, stratify=df['category']
)

cat_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
X_train_vec = cat_vectorizer.fit_transform(X_train)
X_test_vec = cat_vectorizer.transform(X_test)

cat_model = LogisticRegression(max_iter=1000, class_weight='balanced')
cat_model.fit(X_train_vec, y_train)

y_pred = cat_model.predict(X_test_vec)
print(f"Category classifier accuracy: {accuracy_score(y_test, y_pred):.2%}\n")
print(classification_report(y_test, y_pred))

### 4.1 Confusion matrix - Category

In [ ]:
labels = sorted(df['category'].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Category Classifier - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 5. Priority Classification

Same approach, but this time the target is **priority** (High / Medium / Low).

In [ ]:
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    df['cleaned_text'], df['priority'], test_size=0.2, random_state=42, stratify=df['priority']
)

pri_vectorizer = TfidfVectorizer(max_features=3000, ngram_range=(1, 2))
Xp_train_vec = pri_vectorizer.fit_transform(Xp_train)
Xp_test_vec = pri_vectorizer.transform(Xp_test)

pri_model = LogisticRegression(max_iter=1000, class_weight='balanced')
pri_model.fit(Xp_train_vec, yp_train)

yp_pred = pri_model.predict(Xp_test_vec)
print(f"Priority classifier accuracy: {accuracy_score(yp_test, yp_pred):.2%}\n")
print(classification_report(yp_test, yp_pred))

### 5.1 Confusion matrix - Priority

In [ ]:
pri_labels = ['High', 'Medium', 'Low']
cm_p = confusion_matrix(yp_test, yp_pred, labels=pri_labels)

plt.figure(figsize=(5, 4))
sns.heatmap(cm_p, annot=True, fmt='d', cmap='OrRd', xticklabels=pri_labels, yticklabels=pri_labels)
plt.title('Priority Classifier - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 6. Priority Logic: Combining the Model with Business Rules

Real support teams don't rely on the ML model alone for urgency -- they also apply
straightforward rules (e.g. "if the customer says 'urgent' or 'system is down', treat it
as High priority no matter what"). We add that safety net here: if the raw ticket text
contains clear urgency keywords, we bump the predicted priority up to **High**.

In [ ]:
import re

URGENCY_KEYWORDS = [
    "urgent", "urgently", "immediately", "asap", "emergency", "critical",
    "right now", "down", "not working", "cannot access", "can't access",
    "hacked", "unauthorized", "overdue", "deadline", "broken", "crash",
]
PRIORITY_RANK = {"Low": 0, "Medium": 1, "High": 2}

def apply_urgency_boost(text, predicted_priority):
    lowered = text.lower()
    hits = [kw for kw in URGENCY_KEYWORDS if re.search(r'\b' + re.escape(kw) + r'\b', lowered)]
    if hits and PRIORITY_RANK[predicted_priority] < PRIORITY_RANK['High']:
        return 'High', hits
    return predicted_priority, hits

# Example
example_text = "The app keeps crashing and I cannot access my data, please help immediately!"
example_clean = clean_text(example_text)
raw_pred = pri_model.predict(pri_vectorizer.transform([example_clean]))[0]
final_pred, matched_keywords = apply_urgency_boost(example_text, raw_pred)

print("Ticket        :", example_text)
print("Model priority :", raw_pred)
print("Matched keywords:", matched_keywords)
print("Final priority :", final_pred)

## 7. Full Classification Function

This combines category + priority (with the urgency rule) into one function you can
call on any new ticket.

In [ ]:
def classify_ticket(text):
    cleaned = clean_text(text)

    category = cat_model.predict(cat_vectorizer.transform([cleaned]))[0]
    cat_conf = cat_model.predict_proba(cat_vectorizer.transform([cleaned])).max()

    raw_priority = pri_model.predict(pri_vectorizer.transform([cleaned]))[0]
    final_priority, hits = apply_urgency_boost(text, raw_priority)

    return {
        'ticket_text': text,
        'category': category,
        'category_confidence': round(float(cat_conf), 3),
        'priority': final_priority,
        'urgency_keywords_found': hits,
    }

# Try it on a few new, unseen tickets
test_tickets = [
    "My payment failed twice and I was still charged, please refund me urgently!",
    "Can you tell me what colors this product comes in?",
    "The website has been down for an hour and customers cannot check out, please help ASAP.",
    "I'd like to update my email address on my account when you get a chance.",
]

for t in test_tickets:
    result = classify_ticket(t)
    print(f"Ticket   : {result['ticket_text']}")
    print(f"Category : {result['category']} (confidence {result['category_confidence']:.0%})")
    print(f"Priority : {result['priority']}")
    print('-' * 60)

## 8. Save the Trained Models

Saving the models means you don't need to retrain every time -- `src/classify_ticket.py`
loads these files directly so you can classify new tickets instantly from the command line.

In [ ]:
import joblib
os.makedirs('models', exist_ok=True)

joblib.dump(cat_model, 'models/category_model.pkl')
joblib.dump(cat_vectorizer, 'models/category_vectorizer.pkl')
joblib.dump(pri_model, 'models/priority_model.pkl')
joblib.dump(pri_vectorizer, 'models/priority_vectorizer.pkl')

print("Models saved to the models/ folder.")

## 9. Summary

| Model | Accuracy |
|---|---|
| Category classifier | see Section 4 output above |
| Priority classifier | see Section 5 output above |

**What this system does for a support team:**
- Instantly tags incoming tickets with a category, so they route to the right team
- Flags high-priority tickets automatically, so urgent issues aren't missed in a busy queue
- Combines ML predictions with simple, transparent business rules for extra safety on urgency detection

**Next steps if you want to extend this:**
- Swap `data/tickets.csv` for your own historical ticket export
- Try other models (e.g. `LinearSVC`, `RandomForestClassifier`) and compare accuracy
- Add a `subcategory` or `sentiment` label for even more detailed routing
- Wrap `classify_ticket()` in a small API (e.g. Flask/FastAPI) so it can plug into a real helpdesk tool